# 3. Modeling

In [45]:
import pandas as pd

import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeRegressor

In [46]:
RMSE_base = 0.21

## 3.1. Оптимизация препроцессора

**Загрузим данные для обучения**

In [47]:
df = pd.read_csv('E:/ML/housing-prices-ml/data/raw/train.csv')
y = np.log1p(df['SalePrice'])
X = df.drop(columns=['SalePrice', 'Id'])

**Разделим данные на обучающую и тестовыую выборки**

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    train_size=0.8,
    shuffle=True,
    random_state=46)

Разделим признаки на числовые и катгориальные для корректного построения пайплайна обучения


In [49]:
num_features = X.select_dtypes('number').columns.to_list()
cat_features = X.select_dtypes('str').columns.to_list()

Преобразования по результатам EDA

In [50]:
num_features.remove('MSSubClass')
cat_features.append('MSSubClass')

**Рассмотрим влияние гипотез, выдвинутных на этапе EDA**

In [51]:
absence_features = [
    'PoolQC',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
    'GarageFinish',
    'GarageType',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'Alley',
    'Fence',
    'MiscFeature',
    'MasVnrType'
]

regular_cat_features = []

for feature in cat_features:
    if feature not in absence_features:
        regular_cat_features.append(feature)

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

absence_pipline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

regular_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, regular_cat_features),
    ('num', num_pipeline, num_features)
])


Проверим что будет, если числовые признаки с малым количеством числовых значений представить категориальными

In [55]:
for feature in ['MoSold', 'YrSold', 'BsmtHalfBath', 'HalfBath', 'BsmtFullBath', 'FullBath', 'Fireplaces', 'KitchenAbvGr', 'GarageCars', 'BedroomAbvGr', 'TotRmsAbvGrd']:
    for i in range(2):
        test_num_features = num_features.copy()
        test_regular_cat_features = regular_cat_features.copy()
        test_absence_features = absence_features.copy()

        test_num_features.remove(feature)
        if i == 0:
            test_regular_cat_features.append(feature)
        else:
            test_absence_features.append(feature)

        preprocessor = ColumnTransformer([
        ('cat_absence', absence_pipline, absence_features),
        ('cat_reg', regular_cat_pipeline, test_regular_cat_features),
        ('num', num_pipeline, test_num_features)
        ])

        model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeRegressor(random_state=46))
        ])

        cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=46
        )

        cv_scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring='neg_root_mean_squared_error'
        )

        cv_rmse = -cv_scores

        if i == 0:
            print(f'Test: {feature} to regular_cat_features')
        else:
            print(f'Test: {feature} to absence_features')

        print(f'CV RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}\n')

Test: MoSold to regular_cat_features
CV RMSE: 0.21 ± 0.01

Test: MoSold to absence_features
CV RMSE: 0.20 ± 0.01

Test: YrSold to regular_cat_features
CV RMSE: 0.20 ± 0.01

Test: YrSold to absence_features
CV RMSE: 0.21 ± 0.01

Test: BsmtHalfBath to regular_cat_features
CV RMSE: 0.21 ± 0.01

Test: BsmtHalfBath to absence_features
CV RMSE: 0.20 ± 0.01

Test: HalfBath to regular_cat_features
CV RMSE: 0.21 ± 0.01

Test: HalfBath to absence_features
CV RMSE: 0.21 ± 0.02

Test: BsmtFullBath to regular_cat_features
CV RMSE: 0.21 ± 0.02

Test: BsmtFullBath to absence_features
CV RMSE: 0.20 ± 0.01

Test: FullBath to regular_cat_features
CV RMSE: 0.20 ± 0.01

Test: FullBath to absence_features
CV RMSE: 0.20 ± 0.02

Test: Fireplaces to regular_cat_features
CV RMSE: 0.20 ± 0.01

Test: Fireplaces to absence_features
CV RMSE: 0.21 ± 0.02

Test: KitchenAbvGr to regular_cat_features
CV RMSE: 0.21 ± 0.02

Test: KitchenAbvGr to absence_features
CV RMSE: 0.21 ± 0.01

Test: GarageCars to regular_cat_feat

Данные монипуляции не улучшили качество

Сильная линейная связь, замеченная у некоторых числовых признаков не должна оказывать влияния на качество модели, основанных на деревьях

Рассмотрим попмжет ли повысить качество удаление сильно несбалансированных категориальных признаков

In [56]:
for feature in ['Street', 'Alley', 'Utilities', 'LandSlope',
                'Condition2', 'RoofMatl', 'Heating', 'CentralAir',
                'Electrical', 'Functional', 'GarageCond', 'PoolQC',
                'MiscFeature', 'Neighborhood', 'Condition1', 'Condition2',
                'HouseStyle', 'Exterior1st', 'Exterior2nd', 'SaleType']:
    
    test_num_features = num_features.copy()
    test_regular_cat_features = regular_cat_features.copy()
    test_absence_features = absence_features.copy()

    if feature in test_regular_cat_features:
        test_regular_cat_features.remove(feature)
    if feature in test_regular_cat_features:
        test_regular_cat_features.remove(feature)

    preprocessor = ColumnTransformer([
    ('cat_absence', absence_pipline, absence_features),
    ('cat_reg', regular_cat_pipeline, test_regular_cat_features),
    ('num', num_pipeline, test_num_features)
    ])

    model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=46))
    ])

    cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=46
    )

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='neg_root_mean_squared_error'
    )

    cv_rmse = -cv_scores
    
    print(f'Test: REMOVE {feature}')
    print(f'CV RMSE: {cv_rmse.mean():.2f} ± {cv_rmse.std():.2f}\n')

Test: REMOVE Street
CV RMSE: 0.21 ± 0.02

Test: REMOVE Alley
CV RMSE: 0.20 ± 0.02

Test: REMOVE Utilities
CV RMSE: 0.21 ± 0.02

Test: REMOVE LandSlope
CV RMSE: 0.20 ± 0.02

Test: REMOVE Condition2
CV RMSE: 0.21 ± 0.01

Test: REMOVE RoofMatl
CV RMSE: 0.21 ± 0.02

Test: REMOVE Heating
CV RMSE: 0.21 ± 0.02

Test: REMOVE CentralAir
CV RMSE: 0.21 ± 0.02

Test: REMOVE Electrical
CV RMSE: 0.20 ± 0.02

Test: REMOVE Functional
CV RMSE: 0.20 ± 0.02

Test: REMOVE GarageCond
CV RMSE: 0.20 ± 0.02

Test: REMOVE PoolQC
CV RMSE: 0.20 ± 0.02

Test: REMOVE MiscFeature
CV RMSE: 0.20 ± 0.02

Test: REMOVE Neighborhood
CV RMSE: 0.21 ± 0.01

Test: REMOVE Condition1
CV RMSE: 0.21 ± 0.01

Test: REMOVE Condition2
CV RMSE: 0.21 ± 0.01

Test: REMOVE HouseStyle
CV RMSE: 0.20 ± 0.02

Test: REMOVE Exterior1st
CV RMSE: 0.20 ± 0.01

Test: REMOVE Exterior2nd
CV RMSE: 0.20 ± 0.01

Test: REMOVE SaleType
CV RMSE: 0.20 ± 0.01



Данные монипуляции не улучшили качество

## Рассмотрим другие модели